<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/ray_tune_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ray Tune Hyperparameter Sweep

Run hyperparameter tuning sweeps directly on a Colab GPU using [Ray Tune](https://docs.ray.io/en/latest/tune/index.html).

**Key features:**
- **ASHA scheduler** for early stopping of underperforming trials
- **Single-stage sweeps** — tune one curriculum stage at a time
- **Google Drive persistence** — all results, models, and checkpoints saved to Drive
- **Warm-start support** — load a checkpoint from a prior stage or prior sweep

**Recommended runtime:** Colab Pro+ with A100 GPU (more CPU cores for MuJoCo vectorized envs)

**How it compares to Vertex AI sweep:**
| | Vertex AI | Ray Tune (this notebook) |
|---|---|---|
| Scheduling | Bayesian | ASHA early stopping |
| Cost | GCP billing per trial | Included in Colab subscription |
| Parallelism | Multiple cloud workers | Concurrent trials on single GPU |
| Setup | Docker image + GCP project | Just run the notebook |

## 1. Setup & Installation

In [ ]:
import importlib
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Install packages only if not already present
    if importlib.util.find_spec("mujoco") is None:
        get_ipython().system(
            'pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 "stable-baselines3[extra]>=2.2.0" mediapy matplotlib'
        )
    if importlib.util.find_spec("ray") is None:
        get_ipython().system('pip install -q "ray[tune]>=2.9.0"')

    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally — ensure ray[tune], mujoco, stable-baselines3 are installed.")

In [ ]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np

if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = Path("..").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import mujoco

from environments.shared.config import load_all_stages
from environments.shared.scripts.sweep.constants import NET_ARCH_PRESETS
from environments.shared.train_base import SpeciesConfig

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

In [ ]:
# ===== Species & Algorithm =====
SPECIES = "velociraptor"  # @param ["velociraptor", "brachiosaurus", "trex"]
ALGORITHM = "ppo"         # @param ["ppo", "sac"]

# ===== Sweep Stage =====
STAGE = 1                 # @param {type:"integer"} — curriculum stage (1, 2, or 3)

# ===== Ray Tune Budget =====
NUM_TRIALS = 20           # @param {type:"integer"} — total trials to run
MAX_CONCURRENT = 2        # @param {type:"integer"} — concurrent trials (2 for A100, 1 for T4)
TIMESTEPS_PER_TRIAL = 4_000_000  # @param {type:"integer"} — training timesteps per trial

# ===== Training =====
N_ENVS = 4                # @param {type:"integer"} — parallel envs per trial
SEED = 42                 # @param {type:"integer"}
EVAL_FREQ = 50_000        # @param {type:"integer"} — how often to evaluate (also used as ASHA report interval)

# ===== Warm-start (optional) =====
# Path to a model checkpoint to load (e.g. from a prior stage).
# Leave empty to train from scratch.
LOAD_PATH = ""  # @param {type:"string"}

# ===== Google Drive =====
USE_GOOGLE_DRIVE = True   # @param {type:"boolean"}

print(f"Species:    {SPECIES}")
print(f"Algorithm:  {ALGORITHM.upper()}")
print(f"Stage:      {STAGE}")
print(f"Trials:     {NUM_TRIALS} ({MAX_CONCURRENT} concurrent)")
print(f"Timesteps:  {TIMESTEPS_PER_TRIAL:,} per trial")
print(f"Load path:  {LOAD_PATH or '(none — training from scratch)'}")

In [ ]:
# ============================================================
# Google Drive Storage
# ============================================================
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/mesozoic-labs")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Results will persist to: {DRIVE_BASE}")
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
    DRIVE_BASE = repo_root / "logs"
else:
    DRIVE_BASE = repo_root / "logs"
    print(f"Using local storage: {DRIVE_BASE}")

# Create the sweep output directory on Drive
SWEEP_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
SWEEP_DIR = DRIVE_BASE / "ray_tune_sweeps" / SPECIES / f"stage{STAGE}_{ALGORITHM}_{SWEEP_TIMESTAMP}"
SWEEP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Sweep output directory: {SWEEP_DIR}")

## 3. Load Species & Stage Configs

In [ ]:
import importlib

_SPECIES_MAP = {
    "velociraptor": {
        "module": "environments.velociraptor.envs.raptor_env",
        "class": "RaptorEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=strike",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["strike_success", "bite_success"],
    },
    "trex": {
        "module": "environments.trex.envs.trex_env",
        "class": "TRexEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=bite",
        "height_label": "Pelvis height",
        "stage3_section_label": "Hunting",
        "success_keys": ["bite_success", "strike_success"],
    },
    "brachiosaurus": {
        "module": "environments.brachiosaurus.envs.brachio_env",
        "class": "BrachioEnv",
        "stage_descriptions": "1=balance, 2=locomotion, 3=food_reach",
        "height_label": "Torso height",
        "stage3_section_label": "Food Reaching",
        "success_keys": ["food_reached"],
    },
}

assert SPECIES in _SPECIES_MAP, f"Unknown species: {SPECIES}. Choose from: {list(_SPECIES_MAP.keys())}"
_info = _SPECIES_MAP[SPECIES]
_mod = importlib.import_module(_info["module"])
EnvClass = getattr(_mod, _info["class"])

SPECIES_CFG = SpeciesConfig(
    species=SPECIES,
    env_class=EnvClass,
    stage_descriptions=_info["stage_descriptions"],
    height_label=_info["height_label"],
    stage3_section_label=_info["stage3_section_label"],
    success_keys=_info["success_keys"],
)

STAGE_CONFIGS = load_all_stages(SPECIES)

env = EnvClass()
print(f"Environment: {EnvClass.__name__}")
print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.shape}")
for stage_num, cfg in STAGE_CONFIGS.items():
    print(f"  Stage {stage_num}: {cfg['name']} — {cfg['description']}")
env.close()

## 4. Search Space

Translates the same search spaces used by the Vertex AI sweep into Ray Tune format.

**Per-stage rationale (same as Vertex AI notebook):**
- **Stage 1 (balance):** `alive_bonus` is the dominant reward signal — worth sweeping
- **Stage 2 (locomotion):** `alive_bonus` must stay low to avoid standing-trap
- **Stage 3 (behavior):** Only sweep algo params; `alive_bonus` is intentionally small

In [ ]:
from ray import tune

# ── Shared algorithm hyperparameters ──────────────────────────────────────────
_PPO_ALGO_SPACE = {
    "ppo_learning_rate": tune.loguniform(1e-5, 3e-4),
    "ppo_ent_coef": tune.loguniform(1e-4, 0.05),
    "ppo_batch_size": tune.choice([64, 128, 256, 512]),
    "ppo_gamma": tune.uniform(0.97, 0.999),
    "ppo_n_steps": tune.choice([1024, 2048, 4096]),
    "ppo_n_epochs": tune.choice([3, 6, 10]),
    "ppo_net_arch": tune.choice(["small", "medium", "large", "deep", "tapered", "deep_tapered"]),
}

_SAC_ALGO_SPACE = {
    "sac_learning_rate": tune.loguniform(1e-5, 3e-4),
    "sac_batch_size": tune.choice([128, 256, 512]),
    "sac_gamma": tune.uniform(0.97, 0.999),
    "sac_net_arch": tune.choice(["small", "medium", "large", "tapered", "deep_tapered"]),
}

_ALGO_SPACE = _PPO_ALGO_SPACE if ALGORITHM == "ppo" else _SAC_ALGO_SPACE

# ── Per-stage search spaces ────────────────────────────────────────────────────
SEARCH_SPACE_PER_STAGE = {
    1: {
        **_ALGO_SPACE,
        "env_alive_bonus": tune.uniform(1.0, 5.0),
        "env_posture_weight": tune.uniform(0.5, 3.0),
        "env_nosedive_weight": tune.uniform(0.5, 3.0),
    },
    2: {
        **_ALGO_SPACE,
        "env_alive_bonus": tune.uniform(0.1, 1.0),
        "curriculum_warmup_timesteps": tune.choice([50000, 100000, 200000, 300000]),
        "curriculum_warmup_clip_range": tune.uniform(0.01, 0.05),
        "curriculum_warmup_ent_coef": tune.loguniform(0.005, 0.05),
        "curriculum_ramp_timesteps": tune.choice([200000, 500000, 1000000, 2000000]),
        "curriculum_ramp_start_value": tune.uniform(0.05, 0.3),
    },
    3: {
        **_ALGO_SPACE,
        "env_strike_bonus": tune.loguniform(10.0, 100.0),
        "env_strike_approach_weight": tune.uniform(1.0, 5.0),
        "env_strike_proximity_weight": tune.uniform(0.1, 1.0),
        "curriculum_warmup_timesteps": tune.choice([50000, 100000, 200000, 300000]),
        "curriculum_warmup_clip_range": tune.uniform(0.01, 0.05),
        "curriculum_warmup_ent_coef": tune.loguniform(0.005, 0.05),
        "curriculum_ramp_timesteps": tune.choice([500000, 1000000, 2000000, 4000000]),
        "curriculum_ramp_start_value": tune.uniform(0.05, 0.3),
    },
}

SEARCH_SPACE = SEARCH_SPACE_PER_STAGE[STAGE]

print(f"Search space for Stage {STAGE} ({ALGORITHM.upper()}): {len(SEARCH_SPACE)} params")
for name in SEARCH_SPACE:
    print(f"  {name}")

## 5. Trial Training Function

Each Ray Tune trial runs this function. It:
1. Applies the sampled hyperparameters to the TOML stage config
2. Trains using the same `train()` infrastructure as the CLI and Vertex AI
3. Reports `best_mean_reward` to Ray Tune at each evaluation checkpoint (for ASHA)

In [ ]:
import copy
import logging

from ray import train as ray_train
from ray.tune import Callback
from stable_baselines3.common.callbacks import BaseCallback

logger = logging.getLogger(__name__)


class RayTuneReportCallback(BaseCallback):
    """SB3 callback that reports eval metrics to Ray Tune for ASHA scheduling.

    Wraps the SB3 EvalCallback and reports the best mean reward seen so far
    at each evaluation checkpoint. ASHA uses these intermediate reports to
    decide whether to continue or stop a trial early.
    """

    def __init__(self, eval_callback, verbose=0):
        super().__init__(verbose)
        self.eval_callback = eval_callback

    def _on_step(self) -> bool:
        # Check if eval just ran by monitoring last_mean_reward
        if hasattr(self.eval_callback, "last_mean_reward") and self.eval_callback.last_mean_reward is not None:
            ray_train.report({
                "best_mean_reward": float(self.eval_callback.best_mean_reward),
                "last_mean_reward": float(self.eval_callback.last_mean_reward),
                "timesteps": self.num_timesteps,
            })
        return True


def _apply_hpt_config_to_stage(stage_configs, stage, hpt_config, algorithm):
    """Apply Ray Tune sampled hyperparameters to the stage config dict.

    Uses the same naming convention as the Vertex AI sweep:
    - ppo_* / sac_*       -> stage_configs[stage]["ppo_kwargs"] / ["sac_kwargs"]
    - env_*               -> stage_configs[stage]["env_kwargs"]
    - curriculum_*        -> stage_configs[stage]["curriculum_kwargs"]
    - *_net_arch          -> policy_kwargs.net_arch (resolved via NET_ARCH_PRESETS)
    """
    config = stage_configs[stage]
    algo_key = f"{algorithm}_kwargs"

    for key, value in hpt_config.items():
        for prefix in ("ppo", "sac", "env", "curriculum"):
            if key.startswith(prefix + "_"):
                param = key[len(prefix) + 1:]
                if prefix in ("ppo", "sac"):
                    if param == "net_arch":
                        config[algo_key].setdefault("policy_kwargs", {})["net_arch"] = NET_ARCH_PRESETS[value]
                    else:
                        # Cast discrete params to int
                        if param in ("batch_size", "n_steps", "n_epochs"):
                            value = int(value)
                        config[algo_key][param] = value
                elif prefix == "env":
                    config["env_kwargs"][param] = value
                elif prefix == "curriculum":
                    if param in ("warmup_timesteps", "ramp_timesteps"):
                        value = int(value)
                    config["curriculum_kwargs"][param] = value
                break


def train_trial(config):
    """Ray Tune trainable function for a single hyperparameter trial.

    This function runs inside a Ray worker. It reuses the project's existing
    training infrastructure (create_vec_env, SB3 model creation, callbacks)
    rather than calling train() directly, because we need to inject the
    RayTuneReportCallback to report intermediate metrics to ASHA.
    """
    import os
    os.environ["MUJOCO_GL"] = "egl"

    from stable_baselines3 import PPO, SAC
    from stable_baselines3.common.callbacks import (
        CallbackList,
        CheckpointCallback,
        EvalCallback,
    )

    from environments.shared.curriculum import (
        EvalCollapseEarlyStopCallback,
        RewardRampCallback,
        SaveVecNormalizeCallback,
        StageWarmupCallback,
        load_vecnorm_stats,
    )
    from environments.shared.train_base import (
        create_vec_env,
        linear_schedule,
        cosine_schedule,
    )

    # Unpack fixed params from config
    species = config["_species"]
    algorithm = config["_algorithm"]
    stage = config["_stage"]
    timesteps = config["_timesteps"]
    n_envs = config["_n_envs"]
    seed = config["_seed"]
    eval_freq = config["_eval_freq"]
    load_path = config.get("_load_path") or None
    output_dir = config.get("_output_dir")

    # Load species config
    if species == "velociraptor":
        from environments.velociraptor.scripts.train_sb3 import SPECIES_CONFIG
    elif species == "brachiosaurus":
        from environments.brachiosaurus.scripts.train_sb3 import SPECIES_CONFIG
    elif species == "trex":
        from environments.trex.scripts.train_sb3 import SPECIES_CONFIG

    stage_configs = load_all_stages(species)

    # Apply sampled hyperparameters (skip keys starting with _ which are fixed params)
    hpt_params = {k: v for k, v in config.items() if not k.startswith("_")}
    _apply_hpt_config_to_stage(stage_configs, stage, hpt_params, algorithm)

    stage_config = stage_configs[stage]

    # Setup output directory
    trial_id = ray_train.get_context().get_trial_id() or "local"
    trial_dir = Path(output_dir) / trial_id if output_dir else Path(f"/tmp/ray_tune_trial_{trial_id}")
    trial_dir.mkdir(parents=True, exist_ok=True)
    model_dir = trial_dir / "models"
    model_dir.mkdir(exist_ok=True)

    # Create environments
    train_env = create_vec_env(SPECIES_CONFIG, stage_configs, stage, n_envs, seed)
    eval_env = create_vec_env(SPECIES_CONFIG, stage_configs, stage, 1, seed + 1000, use_subproc=False)

    # Load VecNormalize from prior stage
    if load_path:
        vecnorm_path = load_path.replace(".zip", "") + "_vecnorm.pkl"
        if not vecnorm_path.endswith("_vecnorm.pkl"):
            vecnorm_path = load_path + "_vecnorm.pkl"
        if not load_vecnorm_stats(vecnorm_path, train_env, eval_env):
            eval_env.training = False
            eval_env.norm_reward = False
    else:
        eval_env.training = False
        eval_env.norm_reward = False

    # Create or load model
    alg_cls = SAC if algorithm == "sac" else PPO
    algo_key = "sac_kwargs" if algorithm == "sac" else "ppo_kwargs"
    alg_kwargs = stage_config[algo_key].copy()
    alg_kwargs["verbose"] = 0
    alg_kwargs["tensorboard_log"] = str(trial_dir / "tensorboard")

    if algorithm == "ppo":
        lr_end = alg_kwargs.pop("learning_rate_end", None)
        lr_schedule_type = alg_kwargs.pop("lr_schedule", "linear")
        if lr_end is not None:
            lr_start = alg_kwargs["learning_rate"]
            if lr_schedule_type == "cosine":
                alg_kwargs["learning_rate"] = cosine_schedule(lr_start, lr_end)
            else:
                alg_kwargs["learning_rate"] = linear_schedule(lr_start, lr_end)

        clip_range_end = alg_kwargs.pop("clip_range_end", None)
        if clip_range_end is not None:
            clip_start = alg_kwargs["clip_range"]
            alg_kwargs["clip_range"] = linear_schedule(clip_start, clip_range_end)

    policy_kwargs = alg_kwargs.pop("policy_kwargs", None)

    if load_path:
        model = alg_cls.load(load_path, env=train_env, **alg_kwargs)
    else:
        model = alg_cls("MlpPolicy", train_env, policy_kwargs=policy_kwargs, **alg_kwargs)

    # Callbacks
    callbacks = []

    save_vecnorm_cb = SaveVecNormalizeCallback(
        save_path=str(model_dir / "best_model_vecnorm.pkl"),
    )
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(model_dir),
        log_path=str(trial_dir),
        eval_freq=eval_freq // n_envs,
        n_eval_episodes=10,
        deterministic=True,
        render=False,
        verbose=0,
        callback_on_new_best=save_vecnorm_cb,
    )
    callbacks.append(eval_callback)

    # Report to Ray Tune after each eval
    callbacks.append(RayTuneReportCallback(eval_callback))

    # Checkpoint callback
    callbacks.append(CheckpointCallback(
        save_freq=max(eval_freq // n_envs, 1),
        save_path=str(model_dir),
        name_prefix=f"stage{stage}",
        save_vecnormalize=True,
    ))

    # Early stop on reward collapse
    callbacks.append(EvalCollapseEarlyStopCallback(eval_callback=eval_callback, verbose=0))

    # Stage transition callbacks (stages 2+)
    cur_kwargs = stage_config.get("curriculum_kwargs", {})
    if stage > 1 and load_path:
        if algorithm == "ppo":
            callbacks.append(StageWarmupCallback(
                warmup_timesteps=cur_kwargs.get("warmup_timesteps", 100_000),
                warmup_clip_range=cur_kwargs.get("warmup_clip_range", 0.02),
                warmup_ent_coef=cur_kwargs.get("warmup_ent_coef", 0.02),
            ))
        target_fwd_weight = stage_config["env_kwargs"].get("forward_vel_weight", 1.0)
        callbacks.append(RewardRampCallback(
            attr_name="forward_vel_weight",
            start_value=cur_kwargs.get("ramp_start_value", 0.1),
            end_value=target_fwd_weight,
            ramp_timesteps=cur_kwargs.get("ramp_timesteps", 500_000),
        ))

    # Train
    model.learn(
        total_timesteps=timesteps,
        callback=CallbackList(callbacks),
        progress_bar=False,
    )

    # Save final model
    final_path = model_dir / f"stage{stage}_final"
    model.save(str(final_path))
    train_env.save(str(final_path) + "_vecnorm.pkl")

    # Final report
    ray_train.report({
        "best_mean_reward": float(eval_callback.best_mean_reward),
        "timesteps": timesteps,
        "done": True,
    })

    train_env.close()
    eval_env.close()


print("Trial function defined.")

## 6. Run the Sweep

Ray Tune manages the trial scheduling. The **ASHA scheduler** stops underperforming
trials early based on intermediate `best_mean_reward` reports, saving significant compute.

**ASHA parameters:**
- `grace_period`: Minimum number of reports before a trial can be stopped (default: 4 = ~200k timesteps)
- `reduction_factor`: At each rung, keep the top 1/factor trials (default: 3 = keep top ~33%)
- `max_t`: Maximum reports per trial (auto-calculated from timesteps / eval_freq)

In [ ]:
import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler

# Shutdown any existing Ray instance
if ray.is_initialized():
    ray.shutdown()

# Initialize Ray — let it auto-detect resources
ray.init(ignore_reinit_error=True)
print(f"Ray initialized: {ray.cluster_resources()}")

# ASHA scheduler for early stopping
max_reports = TIMESTEPS_PER_TRIAL // EVAL_FREQ
scheduler = ASHAScheduler(
    metric="best_mean_reward",
    mode="max",
    max_t=max_reports,
    grace_period=4,         # Don't stop before 4 eval reports (~200k steps)
    reduction_factor=3,     # Keep top ~33% of trials at each rung
)

# Fixed parameters passed to every trial (prefixed with _ to distinguish from search space)
fixed_config = {
    "_species": SPECIES,
    "_algorithm": ALGORITHM,
    "_stage": STAGE,
    "_timesteps": TIMESTEPS_PER_TRIAL,
    "_n_envs": N_ENVS,
    "_seed": SEED,
    "_eval_freq": EVAL_FREQ,
    "_load_path": LOAD_PATH or "",
    "_output_dir": str(SWEEP_DIR / "trials"),
}

# Merge fixed params with the search space
full_config = {**fixed_config, **SEARCH_SPACE}

print(f"\nStarting Ray Tune sweep:")
print(f"  Species:    {SPECIES}")
print(f"  Stage:      {STAGE}")
print(f"  Algorithm:  {ALGORITHM.upper()}")
print(f"  Trials:     {NUM_TRIALS} ({MAX_CONCURRENT} concurrent)")
print(f"  Timesteps:  {TIMESTEPS_PER_TRIAL:,} per trial")
print(f"  ASHA:       grace_period=4, reduction_factor=3, max_t={max_reports}")
print(f"  Output:     {SWEEP_DIR}")
if LOAD_PATH:
    print(f"  Warm-start: {LOAD_PATH}")

In [ ]:
# Run the sweep
tuner = tune.Tuner(
    train_trial,
    param_space=full_config,
    tune_config=tune.TuneConfig(
        scheduler=scheduler,
        num_samples=NUM_TRIALS,
        max_concurrent_trials=MAX_CONCURRENT,
    ),
    run_config=ray_train.RunConfig(
        name=f"{SPECIES}_stage{STAGE}_{ALGORITHM}",
        storage_path=str(SWEEP_DIR / "ray_results"),
        verbose=1,
    ),
)

results = tuner.fit()
print("\nSweep complete!")

## 7. Results & Analysis

In [ ]:
import pandas as pd

# Get results as a DataFrame
results_df = results.get_dataframe()

# Sort by best reward
if "best_mean_reward" in results_df.columns:
    results_df = results_df.sort_values("best_mean_reward", ascending=False)

# Display the hyperparameter columns (filter out internal Ray columns)
param_cols = [c for c in results_df.columns if c.startswith(("ppo_", "sac_", "env_", "curriculum_"))]
metric_cols = ["best_mean_reward", "last_mean_reward", "timesteps"]
display_cols = [c for c in metric_cols + param_cols if c in results_df.columns]

print(f"\n{'=' * 60}")
print(f"Sweep Results: {SPECIES} Stage {STAGE} ({ALGORITHM.upper()})")
print(f"{'=' * 60}")
display(results_df[display_cols].head(20))

In [ ]:
# Best trial details
best_result = results.get_best_result(metric="best_mean_reward", mode="max")

print(f"\nBest Trial:")
print(f"  best_mean_reward: {best_result.metrics.get('best_mean_reward', 'N/A'):.4f}")
print(f"  Hyperparameters:")
for key, value in best_result.config.items():
    if not key.startswith("_"):
        print(f"    {key}: {value}")

In [ ]:
import matplotlib.pyplot as plt

# Plot reward distribution across all trials
if "best_mean_reward" in results_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of best rewards
    rewards = results_df["best_mean_reward"].dropna()
    axes[0].hist(rewards, bins=min(20, len(rewards)), edgecolor="black", alpha=0.7)
    axes[0].axvline(rewards.max(), color="red", linestyle="--", label=f"Best: {rewards.max():.2f}")
    axes[0].set_xlabel("Best Mean Reward")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"{SPECIES.title()} Stage {STAGE} — Reward Distribution")
    axes[0].legend()

    # Learning rate vs reward scatter (if available)
    lr_col = f"{ALGORITHM}_learning_rate"
    if lr_col in results_df.columns:
        axes[1].scatter(results_df[lr_col], results_df["best_mean_reward"], alpha=0.7)
        axes[1].set_xscale("log")
        axes[1].set_xlabel("Learning Rate")
        axes[1].set_ylabel("Best Mean Reward")
        axes[1].set_title("Learning Rate vs Reward")
    else:
        axes[1].text(0.5, 0.5, "No learning rate data", ha="center", va="center")

    plt.tight_layout()
    plot_path = SWEEP_DIR / "sweep_analysis.png"
    plt.savefig(str(plot_path), dpi=150)
    print(f"Plot saved to: {plot_path}")
    plt.show()

## 8. Save Results to Google Drive

Persist the sweep results CSV, best trial config, and analysis plots to Google Drive
so they survive session restarts.

In [ ]:
# Save results CSV
csv_path = SWEEP_DIR / "sweep_results.csv"
results_df.to_csv(str(csv_path), index=False)
print(f"Results CSV saved to: {csv_path}")

# Save best trial config as JSON
best_config = {k: v for k, v in best_result.config.items() if not k.startswith("_")}
best_config_with_meta = {
    "species": SPECIES,
    "algorithm": ALGORITHM,
    "stage": STAGE,
    "best_mean_reward": best_result.metrics.get("best_mean_reward"),
    "timesteps_per_trial": TIMESTEPS_PER_TRIAL,
    "num_trials": NUM_TRIALS,
    "scheduler": "ASHA",
    "hyperparameters": best_config,
}

best_config_path = SWEEP_DIR / "best_trial_config.json"
with open(str(best_config_path), "w") as f:
    json.dump(best_config_with_meta, f, indent=2, default=str)
print(f"Best trial config saved to: {best_config_path}")

# Copy best trial's model to a convenient location on Drive
best_trial_id = best_result.metrics.get("trial_id", "")
best_trial_model_dir = SWEEP_DIR / "trials" / str(best_trial_id) / "models"
best_model_dest = SWEEP_DIR / "best_model"
best_model_dest.mkdir(parents=True, exist_ok=True)

import shutil

for src_pattern in [f"stage{STAGE}_final.zip", f"stage{STAGE}_final_vecnorm.pkl",
                    "best_model.zip", "best_model_vecnorm.pkl"]:
    src_files = list(best_trial_model_dir.glob(src_pattern)) if best_trial_model_dir.exists() else []
    for src in src_files:
        dest = best_model_dest / src.name
        shutil.copy2(str(src), str(dest))
        print(f"  Copied: {src.name} -> {dest}")

print(f"\nAll results saved to: {SWEEP_DIR}")
print(f"\nTo use the best model for the next stage, set LOAD_PATH to:")
print(f"  {best_model_dest / 'best_model'}")

## 9. Apply Best Hyperparameters

Use this cell to generate `--override` flags for the CLI training scripts,
so you can retrain with the best hyperparameters found by the sweep.

In [ ]:
# Generate CLI override flags for the best trial
overrides = []
for key, value in best_config.items():
    for prefix in ("ppo", "sac", "env", "curriculum"):
        if key.startswith(prefix + "_"):
            param = key[len(prefix) + 1:]
            if param == "net_arch":
                # net_arch is handled specially by the training script
                overrides.append(f"{prefix}.policy_kwargs.net_arch={value}")
            elif isinstance(value, float) and value == int(value):
                overrides.append(f"{prefix}.{param}={int(value)}")
            else:
                overrides.append(f"{prefix}.{param}={value}")
            break

override_str = " ".join(f"--override {o}" for o in overrides)

print(f"Best hyperparameters as CLI overrides:\n")
print(f"python -m environments.{SPECIES}.scripts.train_sb3 train \\")
print(f"    --stage {STAGE} \\")
print(f"    --algorithm {ALGORITHM} \\")
print(f"    --timesteps {TIMESTEPS_PER_TRIAL} \\")
for o in overrides:
    print(f"    --override {o} \\")

## 10. Cleanup

In [ ]:
ray.shutdown()
print("Ray shutdown complete.")
print(f"\nSweep results directory: {SWEEP_DIR}")
print(f"Best trial config: {best_config_path}")
print(f"Best model: {best_model_dest}")